# Submission P07: 0.95816 ≤ aloo/guest ≤ 1.9807

| | |
|---|---|
| Submitted file | `P07.csv` |
| Submitted (UTC) | 2026-10-07 18:15 |
| Public score | 0.93922 (170/181) |
| Rule | 0.95816 ≤ aloo/guest ≤ 1.9807 |

One of the 22 tickets of the final-selection portfolio (E23, `solution/portfolio25.py`): the competition lets a team select up to 25 final submissions and the private leaderboard counts the best of them. Submitted as `P07.csv` from the Kaggle notebook [kacchi-aloo-portfolio](https://www.kaggle.com/code/hosen42/kacchi-aloo-portfolio) v1 (source: `solution/kaggle_portfolio_kernel/`), which writes all 22 tickets.

**Run it:** on Kaggle, upload this notebook and add the competition data (*Add Input → Competitions → The Great Kacchi Aloo Mystery*), then *Save & Run All* and submit `submission.csv`. Locally, run it from this folder (it reads `../data/train.csv` and `../data/test.csv`). The last cell checks the output against the file that was actually submitted.

In [1]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

data: ../data | train 776 rows with aloo_count | test 400 rows


In [2]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [3]:
LO_T, HI_T = 0.95816, 1.9807   # ticket P07: band chosen by the portfolio optimizer (E23)
print(f"band {LO_T} <= aloo/guest <= {HI_T}; train accuracy {(band(tr.apg.values, LO_T, HI_T) == tr.y.values).mean():.4f}")
pred = band(te.apg.values, LO_T, HI_T)

band 0.95816 <= aloo/guest <= 1.9807; train accuracy 0.9420


In [4]:
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": np.asarray(pred).astype(int)})
sub.to_csv("submission.csv", index=False)
fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
print(f"wrote submission.csv: {sub.went_back_for_seconds.sum()} ones / {len(sub)} rows")
print(f"identical to the submitted file 'P07.csv': {fp == 'b16cd9c9c30f616a664c409e3820831f'}")

wrote submission.csv: 181 ones / 400 rows
identical to the submitted file 'P07.csv': True
